# Datathon Task 1 — City Network Traffic Forecasting

Notebook ini melatih model untuk memprediksi speed 1.260 road segment pada horizon **+20, +40, dan +60 menit**. Desainnya sengaja kompetitif tetapi tetap aman dijalankan di Kaggle:

1. **Shared autoregressive ridge** menjadi baseline kuat, cepat, dan selalu menghasilkan submission.
2. **Residual spatio-temporal GNN** mengoreksi baseline menggunakan history, koneksi jalan, identitas road, dan event text.
3. **Validation-weighted ensemble** menggabungkan keduanya per horizon berdasarkan out-of-time validation, bukan bobot tebakan.

**Success criteria:** validation menggunakan tail waktu (bukan random split), tidak memakai target test, notebook berjalan top-to-bottom, dan menghasilkan `/kaggle/working/submission.csv` dengan urutan ID asli.

> Kaggle: pilih accelerator **GPU**. Input default sudah diarahkan ke `/kaggle/input/competitions/datathon-task-1/dataset-task1`. Untuk smoke test lokal, set environment variable `DATATON_FAST_MODE=1`.

## 1. Mengapa pipeline ini?

Traffic forecasting punya tiga komponen berbeda:

- **Temporal inertia:** dalam 20–60 menit, speed biasanya dekat dengan beberapa pembacaan terakhir. Karena itu smoothing/autoregression adalah prior yang sangat kuat.
- **Spatial propagation:** perlambatan pada satu segment dapat menyebar ke jalan terhubung. Adjacency matrix dipakai oleh message passing GNN.
- **Exogenous events:** accident/closure/construction dapat mengubah pola yang tidak terlihat dari speed saja. Teks diubah menjadi fitur TF-IDF low-rank, lalu GNN belajar interaksinya dengan embedding masing-masing road.

Kita memprediksi **residual dari baseline mean-3**, bukan speed dari nol. Ini membuat neural network mulai dari prediksi stabil, lebih mudah dioptimasi, dan mengurangi risiko model deep kalah dari baseline sederhana.

### Mengapa validation harus time-based?

Window train sangat overlap. Random split akan menaruh window yang hampir identik di train dan validation sehingga skor terlihat terlalu bagus. Di sini dua hari terakhir dari setiap blok dijadikan validation, sementara train hanya memakai masa lalu. Gap target secara alami dijaga oleh indeks window.

### Hasil audit lokal sebelum notebook dibuat

Pada tail validation gabungan, persistence sekitar **44,0 MSE**, mean tiga timestep terakhir sekitar **34,4**, shared AR ridge sekitar **30,0**, dan koreksi state low-rank sekitar **29,3**. Seasonal profile harian murni buruk karena distribusi antarblok berubah. Teks linear sendiri nyaris netral, sehingga teks diposisikan sebagai sinyal koreksi di model nonlinear, bukan model utama.

In [ ]:
# Configuration, imports, and reproducibility
from __future__ import annotations

import copy
import gc
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

KAGGLE_INPUT = Path("/kaggle/input/competitions/datathon-task-1/dataset-task1")
LOCAL_CANDIDATES = [Path.cwd() / "dataset-task1"] + [p / "dataset-task1" for p in Path.cwd().parents]
LOCAL_INPUT = next((p for p in LOCAL_CANDIDATES if p.exists()), LOCAL_CANDIDATES[0])
DATA_DIR = KAGGLE_INPUT if KAGGLE_INPUT.exists() else LOCAL_INPUT
IS_KAGGLE = KAGGLE_INPUT.exists()
FAST_MODE = os.getenv("DATATON_FAST_MODE", "0") == "1"

OUTPUT_DIR = Path("/kaggle/working") if IS_KAGGLE else DATA_DIR.parent / "output" / "jupyter-notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = (5, 10, 15)
N_ROADS = 1260
HISTORY = 15
VAL_STEPS = 720 if not FAST_MODE else 120
TRAIN_STRIDE = 2 if not FAST_MODE else 20
TEXT_DIM = 32 if not FAST_MODE else 8
HIDDEN_DIM = 48 if not FAST_MODE else 16
BATCH_SIZE = 16 if torch.cuda.is_available() else (4 if not FAST_MODE else 2)
MAX_EPOCHS = 12 if not FAST_MODE else 1
PATIENCE = 3
NUM_WORKERS = 2 if IS_KAGGLE else 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

print({
    "data_dir": str(DATA_DIR), "output_dir": str(OUTPUT_DIR),
    "device": str(DEVICE), "fast_mode": FAST_MODE,
    "batch_size": BATCH_SIZE, "epochs": MAX_EPOCHS,
})
assert DATA_DIR.exists(), f"Dataset tidak ditemukan di {DATA_DIR}"


## 2. Load dan sanity-check data

Semua assertion di bawah disengaja. Lebih baik notebook berhenti dengan pesan jelas daripada diam-diam menghasilkan submission dengan road/horizon order yang salah.

In [ ]:
def load_json_values(path: Path) -> list[str]:
    with path.open() as f:
        return list(json.load(f).values())


speed_paths = sorted((DATA_DIR / "train").glob("train_speed_*.npy"))
text_paths = sorted((DATA_DIR / "train").glob("train_text_*.json"))
assert len(speed_paths) == len(text_paths) == 2, (speed_paths, text_paths)

# mmap keeps the source arrays cheap in RAM; batches are copied only when needed.
train_speeds = [np.load(p, mmap_mode="r") for p in speed_paths]
train_texts = [load_json_values(p) for p in text_paths]
test_hist = np.load(DATA_DIR / "test" / "test_X_hist.npy")
test_texts = load_json_values(DATA_DIR / "test" / "test_texts.json")
adjacency = np.load(DATA_DIR / "static" / "matrix.npy")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for z, tx in zip(train_speeds, train_texts):
    assert z.ndim == 2 and z.shape[1] == N_ROADS and len(z) == len(tx)
assert test_hist.shape[1:] == (HISTORY, N_ROADS)
assert len(test_hist) == len(test_texts)
assert adjacency.shape == (N_ROADS, N_ROADS)
assert len(sample_submission) == len(test_hist) * len(HORIZONS) * N_ROADS

dataset_summary = pd.DataFrame([
    {
        "split": p.stem,
        "timesteps/samples": len(z),
        "shape": str(z.shape),
        "mean": float(np.mean(z)),
        "std": float(np.std(z)),
        "zero_%": float(100 * np.mean(z == 0)),
        "min": float(np.min(z)),
        "max": float(np.max(z)),
    }
    for p, z in zip(speed_paths, train_speeds)
] + [{
    "split": "test_X_hist", "timesteps/samples": len(test_hist),
    "shape": str(test_hist.shape), "mean": float(test_hist.mean()),
    "std": float(test_hist.std()), "zero_%": float(100 * np.mean(test_hist == 0)),
    "min": float(test_hist.min()), "max": float(test_hist.max()),
}])
display(dataset_summary.round(3))
print("Adjacency edges:", int(adjacency.sum()), "| symmetric:", bool(np.array_equal(adjacency, adjacency.T)))
print("Submission head/tail:")
display(pd.concat([sample_submission.head(2), sample_submission.tail(2)]))


## 3. Exploratory Data Analysis (EDA)

EDA di bawah menjawab pertanyaan yang langsung memengaruhi pipeline:

1. Apakah distribusi speed dan nol konsisten antarblok dan test?
2. Seberapa kuat baseline temporal sederhana pada setiap horizon?
3. Seberapa sparse graph dan berapa banyak tetangga tipikal sebuah road?
4. Apakah event text benar-benar free-form atau memiliki template yang dapat dipadatkan?

Kita tidak menggunakan target test. Test hanya dilihat pada level covariate (history dan text), praktik transductive EDA yang lazim dalam kompetisi.

In [ ]:
# Temporal baseline diagnostics on continuous train series.
baseline_rows = []
for block_name, z in zip([p.stem for p in speed_paths], train_speeds):
    for h in HORIZONS:
        t = np.arange(2, len(z) - h)
        y = np.asarray(z[t + h], dtype=np.float32)
        persistence = np.asarray(z[t], dtype=np.float32)
        mean3 = (np.asarray(z[t], dtype=np.float32) + np.asarray(z[t - 1], dtype=np.float32) + np.asarray(z[t - 2], dtype=np.float32)) / 3
        baseline_rows.extend([
            {"block": block_name, "horizon": f"h{h}", "model": "persistence", "MSE": float(np.mean((y - persistence) ** 2))},
            {"block": block_name, "horizon": f"h{h}", "model": "mean_last_3", "MSE": float(np.mean((y - mean3) ** 2))},
        ])
baseline_eda = pd.DataFrame(baseline_rows)
display(baseline_eda.pivot_table(index=["block", "model"], columns="horizon", values="MSE").round(3))

# Graph diagnostics.
undirected_graph_eda = ((adjacency != 0) | (adjacency.T != 0))
np.fill_diagonal(undirected_graph_eda, False)
degree_eda = undirected_graph_eda.sum(axis=1)
print("Graph degree min/median/mean/p95/max:",
      int(degree_eda.min()), float(np.median(degree_eda)), round(float(degree_eda.mean()), 2),
      float(np.quantile(degree_eda, 0.95)), int(degree_eda.max()))

# Text diagnostics: exact clauses reveal how templated the feed is.
def clauses(text: str) -> list[str]:
    return [x.strip().lower() for x in text.split(".") if x.strip()]

text_eda_rows = []
train_clause_union = set()
for name, docs in zip([p.stem for p in text_paths], train_texts):
    clause_lists = [clauses(s) for s in docs]
    unique = {x for xs in clause_lists for x in xs}
    train_clause_union |= unique
    text_eda_rows.append({
        "split": name, "documents": len(docs), "unique_clauses": len(unique),
        "median_clauses_per_doc": float(np.median([len(x) for x in clause_lists])),
    })
test_clause_lists = [clauses(s) for s in test_texts]
test_clause_set = {x for xs in test_clause_lists for x in xs}
text_eda_rows.append({
    "split": "test_texts", "documents": len(test_texts), "unique_clauses": len(test_clause_set),
    "median_clauses_per_doc": float(np.median([len(x) for x in test_clause_lists])),
})
display(pd.DataFrame(text_eda_rows))
print("Test clauses unseen verbatim in train:", len(test_clause_set - train_clause_union), "/", len(test_clause_set))

# Compact visuals: distribution shift, temporal signal, graph sparsity, and network regime.
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for p, z in zip(speed_paths, train_speeds):
    axes[0, 0].hist(np.asarray(z).mean(axis=0), bins=35, alpha=0.55, label=p.stem)
axes[0, 0].hist(test_hist.mean(axis=(0, 1)), bins=35, alpha=0.55, label="test history")
axes[0, 0].set(title="Mean speed per road", xlabel="km/h", ylabel="roads")
axes[0, 0].legend(fontsize=8)

plot_baseline = baseline_eda.groupby(["model", "horizon"], as_index=False)["MSE"].mean()
for model_name, group in plot_baseline.groupby("model"):
    axes[0, 1].plot(group["horizon"], group["MSE"], marker="o", label=model_name)
axes[0, 1].set(title="Simple temporal baselines", ylabel="MSE")
axes[0, 1].legend()

axes[1, 0].hist(degree_eda, bins=np.arange(degree_eda.max() + 2) - 0.5)
axes[1, 0].set(title="Road graph degree", xlabel="neighbors", ylabel="roads")

for p, z in zip(speed_paths, train_speeds):
    axes[1, 1].plot(np.asarray(z[:720]).mean(axis=1), label=p.stem, linewidth=1)
axes[1, 1].set(title="Network mean speed — first 2 days", xlabel="timestep (4 min)", ylabel="km/h")
axes[1, 1].legend(fontsize=8)
plt.tight_layout()
plt.show()


### Kesimpulan EDA → keputusan preprocessing dan model

| Temuan EDA | Konsekuensi preprocessing/model |
|---|---|
| Mean-last-3 jauh lebih baik daripada persistence dan linear trend mudah meledak | Gunakan smoothing sebagai base forecast; neural model hanya belajar residual |
| Error naik saat horizon makin jauh | Latih direct multi-horizon head, bukan recursive one-step rollout |
| Zero-rate dan mean berbeda antarblok | Jangan random split; gunakan tail tiap blok dan pertahankan zero-mask sebagai fitur |
| Pola harian ada tetapi regime antarblok berubah | Jangan mengandalkan seasonal lookup sebagai model utama |
| Graph sangat sparse, road tipikal hanya punya beberapa tetangga | Gunakan sparse message passing, bukan dense graph multiplication |
| Teks terdiri dari event/location phrases yang berulang, tetapi ada kombinasi baru di test | Gunakan word n-gram TF-IDF + SVD agar kata penyusun masih generalize ke klausa unseen |
| History test tidak identik dengan window train pada audit exact-match | Hindari shortcut lookup; latih model yang benar-benar melakukan forecasting |

Blok pertama berisi 11.160 timestep = 31 × 360; karena 15 timestep = 60 menit, satu hari memang 360 timestep. Informasi ini membantu memahami periodicity, tetapi tidak cukup untuk mengasumsikan kalender test.

## 4. Preprocessing dan honest temporal split

Target untuk history yang berakhir di timestep `t` adalah row `t+5`, `t+10`, dan `t+15`. Dua hari terakhir setiap blok dipakai untuk validation. Window train berakhir sebelum wilayah validation dan sebelum target melewati cut point.

In [ ]:
def valid_times(n: int, start: int, stop: int, stride: int = 1) -> np.ndarray:
    lo = max(HISTORY - 1, start)
    hi = min(stop, n - max(HORIZONS))
    return np.arange(lo, hi, stride, dtype=np.int32)


cuts = [len(z) - VAL_STEPS for z in train_speeds]
train_times = [valid_times(len(z), HISTORY - 1, cut, TRAIN_STRIDE) for z, cut in zip(train_speeds, cuts)]
val_times = [valid_times(len(z), cut, len(z), 1) for z, cut in zip(train_speeds, cuts)]

if FAST_MODE:
    train_times = [t[-min(len(t), 160):] for t in train_times]
    val_times = [t[-min(len(t), 40):] for t in val_times]

print("train windows per block:", [len(x) for x in train_times])
print("validation windows per block:", [len(x) for x in val_times])


def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean((y_true.astype(np.float32) - y_pred.astype(np.float32)) ** 2))


def collect_histories_and_targets(times_by_block):
    histories, targets = [], []
    for z, times in zip(train_speeds, times_by_block):
        histories.extend(np.asarray(z[t - HISTORY + 1:t + 1], dtype=np.float32) for t in times)
        targets.extend(np.stack([z[t + h] for h in HORIZONS], axis=-1).astype(np.float32) for t in times)
    return np.stack(histories), np.stack(targets)


val_hist, val_y = collect_histories_and_targets(val_times)
print("validation tensors:", val_hist.shape, val_y.shape)


### 4.1 Event text: TF-IDF + SVD

Event feed bersifat semi-terstruktur: kalimat seperti `road closure on ...` berulang dengan kombinasi lokasi dan tipe event. TF-IDF word n-gram cocok karena:

- mempertahankan frasa seperti *road closure*, *traffic accident*, dan nama lokasi;
- jauh lebih ringan daripada language model;
- dapat belajar dari 16 ribu dokumen tanpa internet atau model download.

Truncated SVD memadatkan sparse TF-IDF menjadi vektor kecil yang dapat digabung ke GNN. SVD dipasang pada seluruh **train text saja**. Validation text boleh ditransform, tetapi target validation tidak pernah dipakai untuk fitting fitur/model.

In [ ]:
all_train_text = [s for block in train_texts for s in block]
tfidf = TfidfVectorizer(
    ngram_range=(1, 3), min_df=3, max_features=6000,
    sublinear_tf=True, dtype=np.float32,
)
train_text_sparse = tfidf.fit_transform(all_train_text)
n_components = min(TEXT_DIM, max(2, train_text_sparse.shape[1] - 1))
text_svd = TruncatedSVD(n_components=n_components, n_iter=7, random_state=SEED)
all_train_text_z = text_svd.fit_transform(train_text_sparse).astype(np.float32)
test_text_z = text_svd.transform(tfidf.transform(test_texts)).astype(np.float32)

text_offsets = np.cumsum([0] + [len(x) for x in train_texts])
train_text_z = [all_train_text_z[text_offsets[i]:text_offsets[i + 1]] for i in range(len(train_texts))]

print("TF-IDF shape:", train_text_sparse.shape)
print("Text latent shape:", all_train_text_z.shape)
print("Explained variance:", round(float(text_svd.explained_variance_ratio_.sum()), 4))
del train_text_sparse, all_train_text_z
gc.collect()


## 5. Model A — shared autoregressive ridge

Model ini belajar perubahan speed dari 15 lag milik road yang sama. Koefisien dibagi untuk semua road sehingga jutaan pasangan `(time, road)` dapat dipakai tanpa melatih 1.260 model terpisah. Formulasi delta membuat intercept road tidak mendominasi:

\[
\hat y_{t+h,r} = x_{t,r} + \beta_h^\top(x_{t-14:t,r} - x_{t,r})
\]

**Kelebihan:** sangat cepat, tahan overfit, kuat ketika traffic smooth, dan berfungsi sebagai fallback CPU.  
**Kekurangan:** tidak mengetahui road identity, adjacency, atau event text. Karena itu ia dijadikan anchor ensemble, bukan satu-satunya model.

In [ ]:
def fit_shared_ar(times_by_block, max_timestamps_per_block=1000, alpha=1000.0):
    rng = np.random.default_rng(SEED)
    x_parts, y_parts = [], {h: [] for h in HORIZONS}
    for z, times in zip(train_speeds, times_by_block):
        take = min(len(times), max_timestamps_per_block if not FAST_MODE else 60)
        chosen = np.sort(rng.choice(times, size=take, replace=False))
        windows = np.stack([z[t - HISTORY + 1:t + 1] for t in chosen]).astype(np.float32)
        # (sample, road, lag), centered on the latest reading
        features = (windows - windows[:, -1:, :]).transpose(0, 2, 1).reshape(-1, HISTORY)
        x_parts.append(features)
        for h in HORIZONS:
            y_parts[h].append((z[chosen + h] - z[chosen]).reshape(-1).astype(np.float32))
    X = np.concatenate(x_parts)
    models = {}
    for h in HORIZONS:
        y = np.concatenate(y_parts[h])
        models[h] = Ridge(alpha=alpha).fit(X, y)
    return models


def predict_shared_ar(models, histories: np.ndarray) -> np.ndarray:
    n = len(histories)
    features = (histories - histories[:, -1:, :]).transpose(0, 2, 1).reshape(-1, HISTORY)
    latest = histories[:, -1].reshape(-1)
    out = []
    for h in HORIZONS:
        out.append((latest + models[h].predict(features)).reshape(n, N_ROADS))
    return np.stack(out, axis=-1).astype(np.float32)  # sample, road, horizon


ar_models_val = fit_shared_ar(train_times)
val_pred_ar = predict_shared_ar(ar_models_val, val_hist)
ar_scores = {f"h{h}": mse(val_y[..., i], val_pred_ar[..., i]) for i, h in enumerate(HORIZONS)}
ar_scores["mean"] = float(np.mean(list(ar_scores.values())))
print("AR validation MSE:", {k: round(v, 4) for k, v in ar_scores.items()})
print("Learned lag coefficients:")
for h in HORIZONS:
    print(f"h{h}:", np.round(ar_models_val[h].coef_, 3))


## 6. Model B — residual spatio-temporal GraphSAGE

Untuk setiap road, encoder menerima:

- 15 normalized speeds;
- 15 deviations dari mean tiga timestep terakhir;
- 15 indikator apakah speed sama dengan nol;
- learned road embedding;
- compressed event-text vector.

Dua GraphSAGE block lalu menggabungkan hidden state road dengan rata-rata tetangganya. Head menghasilkan koreksi terhadap mean-3 baseline. Last layer diinisialisasi nol, jadi prediksi awal neural network identik dengan baseline stabil—training hanya perlu belajar koreksi yang berguna.

Graph dipakai secara sparse melalui edge list; kita tidak melakukan dense multiplication `1260 × 1260` di setiap batch. Ini penting untuk GPU memory dan runtime Kaggle.

In [ ]:
# Robust undirected neighborhood; self contribution is handled by a separate linear layer.
graph = ((adjacency != 0) | (adjacency.T != 0))
np.fill_diagonal(graph, False)
src_np, dst_np = np.nonzero(graph)
degree_np = np.bincount(dst_np, minlength=N_ROADS).clip(min=1).astype(np.float32)

# Normalization is estimated only from the pre-validation region.
norm_rows = np.concatenate([np.asarray(z[:cut], dtype=np.float32) for z, cut in zip(train_speeds, cuts)])
road_mean = norm_rows.mean(axis=0).astype(np.float32)
road_std = (norm_rows.std(axis=0) + 3.0).astype(np.float32)
del norm_rows
gc.collect()


class TrafficWindowDataset(Dataset):
    def __init__(self, speeds, text_features, times_by_block):
        self.speeds = speeds
        self.text_features = text_features
        self.index = [(b, int(t)) for b, times in enumerate(times_by_block) for t in times]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        b, t = self.index[i]
        z = self.speeds[b]
        hist = np.asarray(z[t - HISTORY + 1:t + 1], dtype=np.float32).T.copy()
        target = np.stack([z[t + h] for h in HORIZONS], axis=-1).astype(np.float32)
        text = np.asarray(self.text_features[b][t], dtype=np.float32)
        return torch.from_numpy(hist), torch.from_numpy(text), torch.from_numpy(target)


class GraphBlock(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.self_linear = nn.Linear(hidden, hidden)
        self.neigh_linear = nn.Linear(hidden, hidden, bias=False)
        self.norm = nn.LayerNorm(hidden)
        self.dropout = nn.Dropout(0.10)

    def forward(self, h, src, dst, degree):
        neigh = torch.zeros_like(h)
        neigh.index_add_(1, dst, h[:, src, :])
        neigh = neigh / degree.view(1, -1, 1)
        update = torch.nn.functional.gelu(self.self_linear(h) + self.neigh_linear(neigh))
        return self.norm(h + self.dropout(update))


class TrafficGNN(nn.Module):
    def __init__(self, road_mean_np, road_std_np, text_dim, hidden=48):
        super().__init__()
        self.register_buffer("road_mean", torch.tensor(road_mean_np).view(1, N_ROADS, 1))
        self.register_buffer("road_std", torch.tensor(road_std_np).view(1, N_ROADS, 1))
        self.register_buffer("src", torch.tensor(src_np, dtype=torch.long))
        self.register_buffer("dst", torch.tensor(dst_np, dtype=torch.long))
        self.register_buffer("degree", torch.tensor(degree_np))
        self.road_embedding = nn.Embedding(N_ROADS, 8)
        self.history_encoder = nn.Sequential(
            nn.Linear(HISTORY * 3 + 8, hidden), nn.GELU(), nn.Dropout(0.10),
            nn.Linear(hidden, hidden), nn.GELU(),
        )
        self.text_encoder = nn.Sequential(nn.Linear(text_dim, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.graph_blocks = nn.ModuleList([GraphBlock(hidden), GraphBlock(hidden)])
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, len(HORIZONS)))
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)

    def forward(self, hist, text):
        # hist: batch, road, time
        normalized = (hist - self.road_mean) / self.road_std
        base = hist[:, :, -3:].mean(dim=-1, keepdim=True)
        deviations = (hist - base) / self.road_std
        zero_mask = (hist == 0).to(hist.dtype)
        road_ids = torch.arange(N_ROADS, device=hist.device)
        road_emb = self.road_embedding(road_ids).unsqueeze(0).expand(len(hist), -1, -1)
        h = self.history_encoder(torch.cat([normalized, deviations, zero_mask, road_emb], dim=-1))
        h = h + self.text_encoder(text).unsqueeze(1)
        for block in self.graph_blocks:
            h = block(h, self.src, self.dst, self.degree)
        residual = self.head(h) * self.road_std
        return base + residual


def make_loaders(times_train, times_valid, shuffle_train=True):
    train_ds = TrafficWindowDataset(train_speeds, train_text_z, times_train)
    val_ds = TrafficWindowDataset(train_speeds, train_text_z, times_valid)
    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=shuffle_train,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), drop_last=False,
    )
    return train_loader, val_loader


In [ ]:
def evaluate_model(model, loader, return_predictions=False):
    model.eval()
    squared_error = 0.0
    count = 0
    predictions = []
    with torch.no_grad():
        for hist, text, target in loader:
            hist, text, target = hist.to(DEVICE), text.to(DEVICE), target.to(DEVICE)
            pred = model(hist, text)
            squared_error += torch.sum((pred - target) ** 2).item()
            count += target.numel()
            if return_predictions:
                predictions.append(pred.cpu().numpy())
    score = squared_error / count
    if return_predictions:
        return score, np.concatenate(predictions)
    return score


def train_model(model, train_loader, val_loader=None, epochs=MAX_EPOCHS, patience=PATIENCE):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    best_state, best_score, best_epoch = None, float("inf"), 0
    history = []
    stale = 0
    for epoch in range(1, epochs + 1):
        started = time.time()
        model.train()
        train_sse, train_count = 0.0, 0
        for hist, text, target in train_loader:
            hist, text, target = hist.to(DEVICE), text.to(DEVICE), target.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(hist, text)
            loss = torch.mean((pred - target) ** 2)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            train_sse += loss.item() * target.numel()
            train_count += target.numel()
        scheduler.step()
        train_mse = train_sse / train_count
        val_mse = evaluate_model(model, val_loader) if val_loader is not None else train_mse
        history.append({"epoch": epoch, "train_mse": train_mse, "val_mse": val_mse})
        print(f"epoch {epoch:02d} | train {train_mse:.4f} | val {val_mse:.4f} | {time.time()-started:.1f}s")
        if val_mse < best_score - 1e-4:
            best_score, best_epoch = val_mse, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if val_loader is not None and stale >= patience:
                print("Early stopping")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, best_score


train_loader, val_loader = make_loaders(train_times, val_times)
gnn_val = TrafficGNN(road_mean, road_std, n_components, HIDDEN_DIM)
gnn_val, training_history, best_epoch, best_val_mse = train_model(gnn_val, train_loader, val_loader)
display(training_history)
_, val_pred_gnn = evaluate_model(gnn_val, val_loader, return_predictions=True)
print("Best GNN epoch/MSE:", best_epoch, round(best_val_mse, 4))


## 7. Ensemble selection dari validation

AR dan GNN membuat error yang berbeda. Bobot optimal dihitung terpisah untuk tiap horizon dengan least squares pada validation:

\[
\hat y = (1-w)\hat y_{AR} + w\hat y_{GNN},\quad 0 \leq w \leq 1
\]

Clipping bobot mencegah extrapolation yang rapuh. Bila GNN tidak menambah informasi untuk horizon tertentu, bobotnya otomatis mendekati nol.

In [ ]:
def optimal_blend_weight(y, a, b):
    direction = (b - a).reshape(-1).astype(np.float64)
    target = (y - a).reshape(-1).astype(np.float64)
    denom = float(direction @ direction)
    return float(np.clip((direction @ target) / max(denom, 1e-12), 0.0, 1.0))


blend_weights = []
blend_rows = []
for i, h in enumerate(HORIZONS):
    w = optimal_blend_weight(val_y[..., i], val_pred_ar[..., i], val_pred_gnn[..., i])
    pred = (1 - w) * val_pred_ar[..., i] + w * val_pred_gnn[..., i]
    blend_weights.append(w)
    blend_rows.append({
        "horizon": f"h{h}", "AR_MSE": mse(val_y[..., i], val_pred_ar[..., i]),
        "GNN_MSE": mse(val_y[..., i], val_pred_gnn[..., i]),
        "GNN_weight": w, "blend_MSE": mse(val_y[..., i], pred),
    })
blend_report = pd.DataFrame(blend_rows)
display(blend_report.round(5))
print("Mean blend MSE:", round(float(blend_report["blend_MSE"].mean()), 5))


## 8. Refit full data dan predict test

Setelah keputusan hyperparameter/bobot dibuat, AR dan GNN dilatih ulang memakai semua label train. GNN full-fit menggunakan jumlah epoch terbaik dari validation stage; minimum satu epoch. Ini menghindari membuang dua hari data per blok.

Jika waktu Kaggle sangat sempit, set `MAX_EPOCHS` lebih kecil di cell konfigurasi atau skip refit GNN dan gunakan AR. Namun submission default di bawah memakai full ensemble.

In [ ]:
full_times = [valid_times(len(z), HISTORY - 1, len(z), TRAIN_STRIDE) for z in train_speeds]
if FAST_MODE:
    full_times = [t[-min(len(t), 200):] for t in full_times]

# Recompute normalization using every train row.
full_rows = np.concatenate([np.asarray(z, dtype=np.float32) for z in train_speeds])
full_road_mean = full_rows.mean(axis=0).astype(np.float32)
full_road_std = (full_rows.std(axis=0) + 3.0).astype(np.float32)
del full_rows
gc.collect()

ar_models_full = fit_shared_ar(full_times)
test_pred_ar = predict_shared_ar(ar_models_full, test_hist)

# A tiny empty validation loader is unnecessary; training score only is printed.
full_loader = DataLoader(
    TrafficWindowDataset(train_speeds, train_text_z, full_times),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(), drop_last=False,
)
full_epochs = max(1, best_epoch)
gnn_full = TrafficGNN(full_road_mean, full_road_std, n_components, HIDDEN_DIM)
gnn_full, full_history, _, _ = train_model(
    gnn_full, full_loader, val_loader=None, epochs=full_epochs, patience=full_epochs + 1,
)


class TestDataset(Dataset):
    def __len__(self):
        return len(test_hist)

    def __getitem__(self, i):
        hist = np.asarray(test_hist[i], dtype=np.float32).T.copy()
        return torch.from_numpy(hist), torch.from_numpy(test_text_z[i])


test_loader = DataLoader(
    TestDataset(), batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
)
gnn_full.eval()
test_pred_gnn_parts = []
with torch.no_grad():
    for hist, text in test_loader:
        pred = gnn_full(hist.to(DEVICE), text.to(DEVICE))
        test_pred_gnn_parts.append(pred.cpu().numpy())
test_pred_gnn = np.concatenate(test_pred_gnn_parts)

w = np.asarray(blend_weights, dtype=np.float32).reshape(1, 1, -1)
test_pred = (1 - w) * test_pred_ar + w * test_pred_gnn
test_pred = np.clip(test_pred, 0.0, 160.0).astype(np.float32)
print("Prediction shape/range/mean:", test_pred.shape, float(test_pred.min()), float(test_pred.max()), float(test_pred.mean()))


## 9. Buat submission dengan order yang benar

Internal prediction berbentuk `(sample, road, horizon)`, sedangkan CSV mengharapkan `(sample, horizon, road)`. Transpose eksplisit dan assertion ID melindungi dari silent ordering bug—salah satu kegagalan paling mahal pada kompetisi seperti ini.

In [ ]:
submission = sample_submission.copy()
flat_prediction = test_pred.transpose(0, 2, 1).reshape(-1)
assert len(flat_prediction) == len(submission)

# Check several boundary IDs before writing.
expected_first = "test_00000_h5_r0"
expected_h10_start = "test_00000_h10_r0"
expected_second_sample = "test_00001_h5_r0"
assert submission.iloc[0]["id"] == expected_first
assert submission.iloc[N_ROADS]["id"] == expected_h10_start
assert submission.iloc[len(HORIZONS) * N_ROADS]["id"] == expected_second_sample

submission["speed"] = flat_prediction
submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print("Rows:", len(submission), "| NaN:", int(submission.speed.isna().sum()))
display(pd.concat([submission.head(3), submission.tail(3)]))


## 10. Pendekatan alternatif dan kapan layak dicoba

| Approach | Mengapa bisa bagus | Mengapa tidak menjadi default notebook ini |
|---|---|---|
| **Persistence / mean-last-k** | Sangat stabil; baseline wajib | Tidak menangkap spatial propagation atau perubahan regime |
| **Per-road AR/Ridge** | Dinamika khusus tiap road | 1.260 model mudah overfit; road dengan banyak nol menjadi tidak stabil |
| **LightGBM/CatBoost global** | Nonlinear, cepat pada fitur tabular; road ID bisa categorical | Dataset hasil ekspansi time × road mencapai >20 juta row dan RAM besar; text/graph interaction perlu feature engineering manual |
| **PCA/low-rank VAR** | Menangkap state kota dan murah | Mode global dapat mengaburkan shock lokal; bagus sebagai komponen ensemble tambahan |
| **STGCN / Graph WaveNet** | Arsitektur mapan untuk traffic graph; receptive field temporal kuat | Implementasi/tuning lebih berat dan rawan runtime; model residual GraphSAGE di sini adalah versi yang lebih sederhana dan robust |
| **DCRNN** | Directed diffusion cocok dengan network jalan | Recurrent decoding jauh lebih lambat untuk 1.260 node |
| **Temporal Transformer** | Dapat memodelkan interaksi panjang dan multivariate | History hanya 15 step; attention antar-1.260 road mahal dan data train relatif kecil |
| **Pretrained language model** | Lebih baik memahami variasi event text | Teks sangat templated; download/model inference mahal, sementara TF-IDF mempertahankan kata event dan lokasi |
| **Road-name event parser** | Bisa memberi fitur event langsung ke road terkait | Metadata roadName berbahasa Mandarin sedangkan event berbahasa Inggris; perlu transliteration/entity map yang tervalidasi |
| **KNN analog traffic state** | Bisa menemukan kondisi kota historis yang mirip | High-dimensional distance sensitif terhadap missing/zero regime dan inference lebih berat |

### Eksperimen berikutnya yang paling bernilai

1. Tambahkan **PCA residual model** sebagai komponen ketiga lalu pilih blend pada temporal validation.
2. Ablation GNN: speed-only vs `+graph` vs `+text` agar kontribusi masing-masing signal terbukti.
3. Coba `TRAIN_STRIDE=1`, hidden 64, dan 15–20 epoch jika GPU/time memungkinkan.
4. Evaluasi split **leave-one-block-out** karena m1 dan m2 memiliki zero-rate/distribusi berbeda.
5. Parse event menjadi `(event_type, location)` multi-hot; ini berpotensi lebih stabil daripada raw clause TF-IDF.

### Risiko yang perlu diingat

- Local validation hanya proxy untuk private test distribution.
- Nol mungkin merepresentasikan closure atau missing sensor; jangan mengimputasi tanpa ablation.
- Jika GNN validation kalah, bobot ensemble otomatis melindungi submission dengan kembali ke AR.
- Jangan mengubah urutan `sample_submission.csv`.